# Brick Kiln Detection — YOLO11-OBB fine-tuning (Kaggle)

Fine-tunes a **pretrained YOLO11-OBB** model on **SentinelKilnDB** (full South Asia) and produces `best.pt`.

### Before you run (Kaggle setup)
1. **Settings -> Accelerator -> GPU** (T4 x2 or P100).
2. **Settings -> Internet -> On** (needed to pip-install and download the dataset).
3. Run cells top to bottom.

### Design decisions (already baked in)
- **Train on the FULL South Asia dataset, evaluate on a Bangladesh-only split.** More data + shared kiln appearance = better accuracy on Bangladesh; the rare CFCBK class has only ~1,944 examples total, so restricting to Bangladesh would break it.
- Classes: `CFCBK`, `FCBK`, `Zigzag` (read from the dataset if it ships a config; otherwise a default is used and printed -- **verify it**).
- Accuracy levers pre-set: rotation + vertical-flip augmentation (correct for overhead imagery), cosine LR, early stopping, TTA at eval.

### Time / accuracy
`yolo11s-obb` @ imgsz 256 with early-stopping takes a few hours on Kaggle GPU. Bump `MODEL` to `m`/`l` for more accuracy (slower); drop to `n` or imgsz 128 for speed. If you hit Kaggle's session limit, lower `EPOCHS`/`IMGSZ` or set `RESUME=True` to continue.

Output: `/kaggle/working/best.pt` -- hand this one file to whoever builds the rest.


In [1]:
# 1. Install + imports + GPU check
!pip -q install -U ultralytics huggingface_hub pandas pyarrow

import os, re, glob, shutil
from pathlib import Path
import torch
from ultralytics import YOLO

print("Ultralytics OK | CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - enable it in Settings")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 84.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.3 whi

In [2]:
# 2. CONFIG - tune these
HF_REPO   = "SustainabilityLabIITGN/SentinelKilnDB"   # dataset on Hugging Face
MODEL     = "yolo11n-obb.pt"   # accuracy: yolo11m-obb.pt / yolo11l-obb.pt   |  speed: yolo11n-obb.pt
IMGSZ     = 128               # native tiles are 128px; 256 usually helps small-object OBB (experiment)
EPOCHS    = 50                # upper bound; early stopping will likely stop sooner
BATCH     = 256                # lower to 16 if you get CUDA out-of-memory
PATIENCE  = 15                 # early-stopping patience
DEVICE    = "0,1"                 # single GPU. On Kaggle T4 x2 you can use "0,1" for multi-GPU
RESUME    = False              # set True to continue an interrupted run
RUN_NAME  = "kiln_yolo11"

# Bangladesh bounding box for the region-specific eval (approx)
BD_LAT = (20.5, 26.7)
BD_LON = (88.0, 92.7)

WORK     = Path("/kaggle/working")
RAW_DIR  = WORK / "skdb_raw"      # downloaded dataset
DATA_DIR = WORK / "skdb_yolo"     # assembled YOLO-format data
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# 3. Get the data
# OPTION A (recommended if a mirror exists): click 'Add Input' in the Kaggle sidebar, search
#   'SentinelKilnDB', attach it, and set RAW_DIR = Path('/kaggle/input/<the-dataset-slug>').
#   Then SKIP this download and set local = str(RAW_DIR) below.
# OPTION B: download from Hugging Face (needs Internet = On). Runs below.

from huggingface_hub import snapshot_download, list_repo_files

files = list_repo_files(HF_REPO, repo_type="dataset")
print("Repo has", len(files), "files. First 40:")
for f in files[:40]:
    print("  ", f)

# Download everything (tiles are tiny; total is a couple of GB). To limit, pass allow_patterns=[...]
local = snapshot_download(HF_REPO, repo_type="dataset", local_dir=str(RAW_DIR))
print("\nDownloaded to:", local)


Repo has 5 files. First 40:
   .gitattributes
   README.md
   test/test.parquet
   train/train.parquet
   val/val.parquet


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]


Downloaded to: /kaggle/working/skdb_raw


In [4]:
# 4. INSPECT what we got  --  this drives everything below. Read the printout.
raw = Path(local)

yamls    = sorted(raw.rglob("*.yaml")) + sorted(raw.rglob("*.yml"))
classfs  = sorted(raw.rglob("classes.txt")) + sorted(raw.rglob("*.names"))
parquets = sorted(raw.rglob("*.parquet"))
img_dirs = [p for p in raw.rglob("images") if p.is_dir()]
lbl_dirs = [p for p in raw.rglob("labels") if p.is_dir()]

print("YAML configs :", [str(p.relative_to(raw)) for p in yamls][:5])
print("class files  :", [str(p.relative_to(raw)) for p in classfs][:5])
print("parquet files:", len(parquets), [str(p.relative_to(raw)) for p in parquets[:3]])
print("images/ dirs :", [str(p.relative_to(raw)) for p in img_dirs][:8])
print("labels/ dirs :", [str(p.relative_to(raw)) for p in lbl_dirs][:8])

# If parquet exists, peek at its columns so extraction targets the right fields
if parquets:
    import pandas as pd
    _df = pd.read_parquet(parquets[0])
    print("\nParquet columns:", _df.columns.tolist())
    print("First row types:", {c: type(_df.iloc[0][c]).__name__ for c in _df.columns})


YAML configs : []
class files  : []
parquet files: 3 ['test/test.parquet', 'train/train.parquet', 'val/val.parquet']
images/ dirs : []
labels/ dirs : []

Parquet columns: ['image_name', 'image', 'dota_label', 'yolo_aa_label', 'yolo_obb_label']
First row types: {'image_name': 'str', 'image': 'bytes', 'dota_label': 'ndarray', 'yolo_aa_label': 'ndarray', 'yolo_obb_label': 'ndarray'}


In [5]:
# 5 (FINAL — labels are ready-made YOLO-OBB strings). Re-run this over the previous cell 5.
import pyarrow.parquet as pqt

pq = {p.name.split(".")[0]: p for p in parquets}   # {'train':.., 'val':.., 'test':..}

def parquet_to_yolo(pq_path, split):
    idir = DATA_DIR/split/"images"; ldir = DATA_DIR/split/"labels"
    idir.mkdir(parents=True, exist_ok=True); ldir.mkdir(parents=True, exist_ok=True)
    counts, total = {}, 0
    for b in pqt.ParquetFile(pq_path).iter_batches(batch_size=512):
        for _, row in b.to_pandas().iterrows():
            name = str(row["image_name"]).rsplit(".", 1)[0]
            png = idir/(name + ".png")
            if not png.exists():                          # images already extracted -> skip, stay fast
                with open(png, "wb") as f: f.write(row["image"])
            lab = row["yolo_obb_label"]                   # object array; each item is a full OBB line
            lines = [str(s).strip() for s in lab if str(s).strip()]
            for ln in lines:
                c = int(ln.split()[0]); counts[c] = counts.get(c, 0) + 1
            open(ldir/(name + ".txt"), "w").write("\n".join(lines))
            total += 1
    return total, counts

print("Writing train labels ..."); n_tr, c_tr = parquet_to_yolo(pq["train"], "train")
print("Writing val labels ...");   n_va, c_va = parquet_to_yolo(pq["val"],   "val")
print("train tiles:", n_tr, "| val tiles:", n_va)
print("train class-index counts:", dict(sorted(c_tr.items())))    # <-- should now be NON-empty

# Class names by frequency (paper: FCBK most common, then Zigzag, CFCBK rarest)
rank = ["FCBK", "Zigzag", "CFCBK"]
by_freq = sorted(c_tr, key=lambda k: c_tr[k], reverse=True)
name_by_idx = {idx: rank[r] for r, idx in enumerate(by_freq)}
CLASS_NAMES = [name_by_idx[i] for i in sorted(name_by_idx)]
print(">>> class index -> name:", name_by_idx)
print(">>> CLASS_NAMES =", CLASS_NAMES)

DATA_YAML = DATA_DIR/"data.yaml"
DATA_YAML.write_text("path: %s\ntrain: train/images\nval: val/images\nnc: %d\nnames: %s\n"
                     % (DATA_DIR, len(CLASS_NAMES), CLASS_NAMES))
print("\n", DATA_YAML.read_text())

Writing train labels ...
Writing val labels ...
train tiles: 71856 | val tiles: 23952
train class-index counts: {0: 2032, 1: 34292, 2: 27463}
>>> class index -> name: {1: 'FCBK', 2: 'Zigzag', 0: 'CFCBK'}
>>> CLASS_NAMES = ['CFCBK', 'FCBK', 'Zigzag']

 path: /kaggle/working/skdb_yolo
train: train/images
val: val/images
nc: 3
names: ['CFCBK', 'FCBK', 'Zigzag']



In [6]:
# 6 (CORRECTED): Bangladesh-only eval carved from the VAL split
val_img, val_lbl = DATA_DIR/"val"/"images", DATA_DIR/"val"/"labels"
BD_DIR = DATA_DIR/"bd_val"
(BD_DIR/"images").mkdir(parents=True, exist_ok=True); (BD_DIR/"labels").mkdir(parents=True, exist_ok=True)

num_re = re.compile(r"(-?\d{1,3}\.\d+)")
def coords_from(name):
    n = num_re.findall(name)
    if len(n) >= 2:
        a, b = float(n[0]), float(n[1]); return (a, b) if 5 < abs(a) < 40 else (b, a)
    return None

kept = 0
for img in val_img.glob("*"):
    c = coords_from(img.name)
    if not c: continue
    lat, lon = c
    if BD_LAT[0] <= lat <= BD_LAT[1] and BD_LON[0] <= lon <= BD_LON[1]:
        for src, dstd in [(img, BD_DIR/"images"), (val_lbl/(img.stem + ".txt"), BD_DIR/"labels")]:
            if src.exists() and not (dstd/src.name).exists():
                try: os.symlink(src, dstd/src.name)
                except FileExistsError: pass
        kept += 1

if kept:
    BD_YAML = DATA_DIR/"data_bd.yaml"
    BD_YAML.write_text("path: %s\ntrain: images\nval: images\nnc: %d\nnames: %s\n"
                       % (BD_DIR, len(CLASS_NAMES), CLASS_NAMES))
    print("Bangladesh eval tiles:", kept, "->", BD_YAML)
else:
    BD_YAML = None
    print("No BD tiles matched (image_names may not be coordinates). Overall val still runs.")

Bangladesh eval tiles: 3929 -> /kaggle/working/skdb_yolo/data_bd.yaml


In [7]:
# 7. TRAIN
model = YOLO(MODEL)   # downloads pretrained YOLO11-OBB weights the first time
results = model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE,
    device=DEVICE, project=str(WORK), name=RUN_NAME, resume=RESUME,
    optimizer="auto", cos_lr=True, close_mosaic=10,
    # augmentation - rotation + vertical flip are the aerial/OBB-specific wins:
    degrees=90.0, fliplr=0.5, flipud=0.5, scale=0.5, translate=0.1, mosaic=1.0,
    cache=True, plots=True, verbose=True,
)
print("Training done. Best weights:", results.save_dir)


Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=256, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/skdb_yolo/data.yaml, degrees=90.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=128, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=kiln_

AttributeError: 'dict' object has no attribute 'save_dir'

In [ ]:
# 8. EVALUATE - overall, then Bangladesh-only, with test-time augmentation (TTA)
best_path = Path(WORK)/RUN_NAME/"weights"/"best.pt"
best = YOLO(str(best_path))

print("=== Overall (training regions) ===")
m_all = best.val(data=str(DATA_YAML), imgsz=IMGSZ, augment=True)
print(m_all.results_dict)

if BD_YAML:
    print("\n=== Bangladesh-only ===")
    m_bd = best.val(data=str(BD_YAML), imgsz=IMGSZ, augment=True)
    print(m_bd.results_dict)


In [ ]:
# 9. Hand off best.pt
final = Path(WORK)/"best.pt"
shutil.copy(best_path, final)
print("Saved:", final, "(", round(final.stat().st_size/1e6, 1), "MB )")

print("NEXT STEPS")
print("- Download it from the Kaggle 'Output' tab of this notebook (best.pt).")
print("- Or push to Hugging Face for your teammate (needs your HF token):")
print("    from huggingface_hub import HfApi")
print("    HfApi().upload_file(path_or_fileobj='/kaggle/working/best.pt',")
print("                        path_in_repo='best.pt',")
print("                        repo_id='<your-username>/kiln-detector',")
print("                        repo_type='model', token='hf_...')")
print("- Quick sanity test on one image:")
print("    from ultralytics import YOLO")
print("    YOLO('/kaggle/working/best.pt').predict('path/to/a_tile.png', save=True, augment=True)")
